# 3-Channel Bayesian Confidence Model — Explorer

## What this model does

Given a spectrum and its top-1 library match (ranked by entropy similarity), this model
answers: **how confident are we that this match is correct?**

It combines three independent evidence channels via Bayes' rule:

```
posterior odds = prior odds × LR_entropy_sim × LR_sim_gap × LR_delta_rt
```

Each likelihood ratio (LR) asks: "how much more likely is this feature value under the
correct-match hypothesis vs. the wrong-match hypothesis?"

## Key files
- `bayesian_score_v2.py` — scoring engine (ChannelSpec, FittedChannel, logit transforms)
- `build_features_v2.py` — builds `data/feature_table_v2.csv` (65K hit-level rows)
- `channel_eda_v2.ipynb` — per-channel distribution EDA

## Validated performance (2026-04-17)
- **AUC = 0.841** [bootstrap 95% CI: 0.830–0.852]
- 5-fold GroupKFold by InChIKey-14, no group leakage
- FDR @ conf ≥ 0.9: 4.9%

In [ ]:
import sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import GroupKFold

sys.path.insert(0, '.')
from bayesian_score_v2 import (
    ChannelSpec, FittedChannel, fit_channel,
    compute_logLR_table, compute_posterior,
    logit_upper_half
)

warnings.filterwarnings('ignore')
np.random.seed(42)

# ── Load feature table and select top-1 candidate per spectrum ──
ft = pd.read_csv('../data/feature_table_v2.csv')

top1 = (ft.sort_values('entropy_similarity', ascending=False)
          .groupby('wiki_id').first().reset_index())

# Fill missing RT with 0 (neutral — contributes 0 logLR)
top1['signed_delta_rt'] = top1['signed_delta_rt'].fillna(0)

labels = top1['hit_label'].values
prior_tp = labels.mean()

print(f'Spectra:          {len(top1):,}')
print(f'Correct top-1:    {labels.sum():,} ({prior_tp:.1%})')
print(f'Wrong top-1:      {(labels==0).sum():,} ({1-prior_tp:.1%})')
print(f'  yy_ (FP):       {(top1["spectrum_label"]=="FP").sum():,}')
print(f'  TP wrong rank-1: {((top1["spectrum_label"]=="TP") & (labels==0)).sum():,}')

## Step 1 — Define the three channels

Each `ChannelSpec` declares:
- Which column in the feature table to use
- What parametric family to fit for TP and FP (Normal, Student-t, etc.)
- An optional transform applied before fitting (logit for entropy_sim)
- A clamp range `[-5, +5]` so no single channel can dominate the total logLR

In [ ]:
CHANNELS = [
    # Channel 1: spectral match quality
    #   logit_upper_half maps [0.5, 1.0] → ℝ, capped at ±4.
    #   Without the cap, esim=1.0 → logit=13, far outside the fitted Normal.
    ChannelSpec('entropy_sim', 'entropy_similarity', 'continuous',
                higher_means_tp=True, tp_family='normal', fp_family='normal',
                transform=logit_upper_half),

    # Channel 2: margin over next-best candidate
    #   Large gap → this candidate clearly stands out.
    ChannelSpec('sim_gap', 'sim_gap', 'continuous',
                higher_means_tp=True, tp_family='normal', fp_family='normal'),

    # Channel 3: retention time prediction error (signed, seconds)
    #   Student-t (df=3) because RT errors have heavy tails
    #   (e.g. N-acetyl amino acids have systematically wrong predicted RT).
    ChannelSpec('signed_delta_rt', 'signed_delta_rt', 'continuous',
                higher_means_tp=True, tp_family='student_t', fp_family='student_t'),
]

print('Channels defined:')
for c in CHANNELS:
    print(f'  {c.name:20s}  col={c.feature_col:25s}  TP={c.tp_family:10s}  FP={c.fp_family:10s}  '
          f'transform={"logit_upper_half" if c.transform else "none":20s}  clamp={c.clamp}')

## Step 2 — Fit TP/FP distributions on all data

For each channel, split the data by `hit_label` and fit the declared parametric family.
This gives us the densities `p(x | TP)` and `p(x | FP)` that power the likelihood ratios.

We fit on the full dataset first to inspect the distributions; cross-validation comes later.

In [ ]:
# Fit all channels on full data
fitted_channels = []
for spec in CHANNELS:
    fc = fit_channel(spec, top1[spec.feature_col].values, labels)
    fitted_channels.append(fc)

# Print fitted parameters
for fc in fitted_channels:
    name = fc.spec.name
    tp_d, fp_d = fc.tp_dist, fc.fp_dist
    print(f'--- {name} (TP n={fc.tp_n}, FP n={fc.fp_n}) ---')
    if fc.spec.tp_family == 'normal':
        print(f'  TP ~ Normal(mu={tp_d.mean():.3f}, sigma={tp_d.std():.3f})')
        print(f'  FP ~ Normal(mu={fp_d.mean():.3f}, sigma={fp_d.std():.3f})')
    elif fc.spec.tp_family == 'student_t':
        print(f'  TP ~ Student-t(df=3, loc={tp_d.kwds.get("loc", tp_d.args[1]):.3f}, '
              f'scale={tp_d.kwds.get("scale", tp_d.args[2]):.3f})')
        print(f'  FP ~ Student-t(df=3, loc={fp_d.kwds.get("loc", fp_d.args[1]):.3f}, '
              f'scale={fp_d.kwds.get("scale", fp_d.args[2]):.3f})')
    print()

## Step 3 — Visualize per-channel distributions and logLR curves

Three panels per channel:
1. **Left**: TP vs FP histograms with fitted density overlaid — shows separation
2. **Center**: log-likelihood ratio curve — where the channel provides evidence
3. **Right**: Clamped logLR distribution by class — what the model actually sees

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(18, 14))

for row, fc in enumerate(fitted_channels):
    col_name = fc.spec.feature_col
    raw_vals = top1[col_name].values
    tp_raw = raw_vals[labels == 1]
    fp_raw = raw_vals[labels == 0]

    # Apply transform for display (entropy_sim → logit scale)
    if fc.spec.transform is not None:
        tp_t = fc.spec.transform(tp_raw.copy())
        fp_t = fc.spec.transform(fp_raw.copy())
        xlabel = f'{fc.spec.name} (logit-transformed)'
    else:
        tp_t = tp_raw[np.isfinite(tp_raw)]
        fp_t = fp_raw[np.isfinite(fp_raw)]
        xlabel = fc.spec.name

    # ── Panel 1: TP vs FP histograms + fitted densities ──
    ax = axes[row, 0]
    lo = np.nanpercentile(np.concatenate([tp_t, fp_t]), 1)
    hi = np.nanpercentile(np.concatenate([tp_t, fp_t]), 99)
    grid = np.linspace(lo, hi, 500)

    ax.hist(tp_t, bins=60, density=True, alpha=0.4, color='steelblue',
            label=f'TP (n={len(tp_t):,})', range=(lo, hi))
    ax.hist(fp_t, bins=60, density=True, alpha=0.4, color='salmon',
            label=f'FP (n={len(fp_t):,})', range=(lo, hi))

    # Overlay fitted PDFs
    ax.plot(grid, np.exp(fc.tp_dist.logpdf(grid)), 'b-', lw=2, label='TP fit')
    ax.plot(grid, np.exp(fc.fp_dist.logpdf(grid)), 'r-', lw=2, label='FP fit')
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Density')
    ax.set_title(f'{fc.spec.name}: TP vs FP distributions')
    ax.legend(fontsize=8)

    # ── Panel 2: logLR curve ──
    ax = axes[row, 1]
    tp_pdf = np.exp(fc.tp_dist.logpdf(grid))
    fp_pdf = np.exp(fc.fp_dist.logpdf(grid))
    raw_logLR = np.log(np.maximum(tp_pdf, 1e-300)) - np.log(np.maximum(fp_pdf, 1e-300))
    clamped_logLR = np.clip(raw_logLR, *fc.spec.clamp)

    ax.plot(grid, raw_logLR, 'gray', lw=1, alpha=0.5, label='raw logLR')
    ax.plot(grid, clamped_logLR, 'darkorange', lw=2, label='clamped [-5, +5]')
    ax.axhline(0, color='black', linestyle=':', lw=0.8)
    ax.fill_between(grid, 0, clamped_logLR,
                     where=clamped_logLR > 0, alpha=0.15, color='steelblue', label='favors TP')
    ax.fill_between(grid, 0, clamped_logLR,
                     where=clamped_logLR < 0, alpha=0.15, color='salmon', label='favors FP')
    ax.set_xlabel(xlabel)
    ax.set_ylabel('log LR')
    ax.set_title(f'{fc.spec.name}: likelihood ratio curve')
    ax.legend(fontsize=8)

    # ── Panel 3: clamped logLR distributions by class ──
    ax = axes[row, 2]
    lr_all = fc.logLR(raw_vals)
    lr_tp = lr_all[labels == 1]
    lr_fp = lr_all[labels == 0]

    ax.hist(lr_tp, bins=60, density=True, alpha=0.5, color='steelblue', label=f'TP logLR')
    ax.hist(lr_fp, bins=60, density=True, alpha=0.5, color='salmon', label=f'FP logLR')
    ax.axvline(0, color='black', linestyle=':', lw=0.8)
    ax.set_xlabel('Clamped logLR')
    ax.set_ylabel('Density')
    ax.set_title(f'{fc.spec.name}: logLR by class')
    ax.legend(fontsize=8)

    # Univariate AUC for this channel
    auc_ch = roc_auc_score(labels, lr_all)
    ax.text(0.98, 0.95, f'AUC={auc_ch:.3f}', transform=ax.transAxes,
            ha='right', va='top', fontsize=10, fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.tight_layout()
plt.show()

## Step 4 — Combine channels: from logLR to posterior

The magic of Bayesian scoring in log-odds form:

```
log_posterior_odds = log_prior_odds + logLR_ch1 + logLR_ch2 + logLR_ch3
posterior = sigmoid(log_posterior_odds)
```

This works because under channel independence, the joint likelihood ratio factors:
`p(x1,x2,x3|TP) / p(x1,x2,x3|FP) = LR1 × LR2 × LR3`, which is addition in log-space.

The prior (68.7% of top-1 are correct) sets the baseline. Each channel then shifts
the log-odds up (toward TP) or down (toward FP).

In [ ]:
# Score all spectra using the full-data fit (not cross-validated — just for exploration)
total_logLR = np.zeros(len(top1))
per_channel_logLR = {}

for fc in fitted_channels:
    lr = fc.logLR(top1[fc.spec.feature_col].values)
    per_channel_logLR[fc.spec.name] = lr
    total_logLR += lr

# Convert to posterior
prior_log_odds = np.log(prior_tp / (1 - prior_tp))
posterior = 1.0 / (1.0 + np.exp(-(prior_log_odds + total_logLR)))

# Stacked contribution visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── Panel 1: waterfall — how each channel shifts log-odds ──
ax = axes[0]
ch_names = [fc.spec.name for fc in fitted_channels]
tp_means = [per_channel_logLR[n][labels == 1].mean() for n in ch_names]
fp_means = [per_channel_logLR[n][labels == 0].mean() for n in ch_names]

x = np.arange(len(ch_names))
w = 0.35
ax.bar(x - w/2, tp_means, w, color='steelblue', label='TP mean logLR')
ax.bar(x + w/2, fp_means, w, color='salmon', label='FP mean logLR')
ax.set_xticks(x)
ax.set_xticklabels(ch_names)
ax.axhline(0, color='black', linestyle=':', lw=0.8)
ax.set_ylabel('Mean logLR contribution')
ax.set_title('Per-channel evidence (mean)')
ax.legend()

# ── Panel 2: total logLR distribution ──
ax = axes[1]
ax.hist(total_logLR[labels == 1], bins=60, density=True, alpha=0.5,
        color='steelblue', label='TP')
ax.hist(total_logLR[labels == 0], bins=60, density=True, alpha=0.5,
        color='salmon', label='FP')
ax.axvline(0, color='black', linestyle=':', lw=0.8)
ax.set_xlabel('Total logLR (sum of 3 channels)')
ax.set_ylabel('Density')
ax.set_title('Combined logLR by class')
ax.legend()

auc_total = roc_auc_score(labels, total_logLR)
ax.text(0.98, 0.95, f'AUC={auc_total:.3f}', transform=ax.transAxes,
        ha='right', va='top', fontsize=11, fontweight='bold',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

# ── Panel 3: posterior distribution ──
ax = axes[2]
ax.hist(posterior[labels == 1], bins=60, density=True, alpha=0.5,
        color='steelblue', label='TP')
ax.hist(posterior[labels == 0], bins=60, density=True, alpha=0.5,
        color='salmon', label='FP')
ax.axvline(prior_tp, color='black', linestyle='--', lw=1, label=f'prior={prior_tp:.2f}')
ax.set_xlabel('Posterior P(correct)')
ax.set_ylabel('Density')
ax.set_title('Posterior confidence by class')
ax.legend()

plt.tight_layout()
plt.show()

print(f'Prior log-odds: {prior_log_odds:.3f}  (prior = {prior_tp:.3f})')
print(f'Total logLR — TP mean: {total_logLR[labels==1].mean():+.2f}  '
      f'FP mean: {total_logLR[labels==0].mean():+.2f}')
print(f'Posterior — TP median: {np.median(posterior[labels==1]):.3f}  '
      f'FP median: {np.median(posterior[labels==0]):.3f}')

## Step 5 — Cross-validated AUC (the real number)

The AUC above is optimistic because we fitted and scored on the same data.
The honest evaluation uses **5-fold GroupKFold** grouped by `anno_ik14` (InChIKey-14):

- All spectra of the same compound go into the same fold
- No compound appears in both train and test
- This prevents the model from memorizing compound-specific distribution quirks

In [ ]:
# Build groups for GroupKFold — assign unique fake IDs where anno_ik14 is missing
groups = top1['anno_ik14'].fillna('').values.copy()
for i in range(len(groups)):
    if groups[i] == '':
        groups[i] = f'__no_ik14_{i}'

gkf = GroupKFold(n_splits=5)
oof_posterior = np.full(len(top1), np.nan)
oof_logLR = np.full(len(top1), np.nan)
fold_aucs = []

for fold, (train_idx, test_idx) in enumerate(gkf.split(top1, labels, groups)):
    train_df = top1.iloc[train_idx]
    test_df = top1.iloc[test_idx]
    train_labels = labels[train_idx]
    test_labels = labels[test_idx]

    # Fit on train fold only
    fitted = []
    for spec in CHANNELS:
        fc = fit_channel(spec, train_df[spec.feature_col].values, train_labels)
        if fc is not None:
            fitted.append(fc)

    # Score test fold
    fold_logLR = np.zeros(len(test_df))
    for fc in fitted:
        fold_logLR += fc.logLR(test_df[fc.spec.feature_col].values)

    fold_post = 1.0 / (1.0 + np.exp(-(prior_log_odds + fold_logLR)))
    oof_posterior[test_idx] = fold_post
    oof_logLR[test_idx] = fold_logLR

    fold_auc = roc_auc_score(test_labels, fold_post)
    fold_aucs.append(fold_auc)
    print(f'Fold {fold}: n={len(test_idx):,}  TP={test_labels.sum():,}  '
          f'FP={(test_labels==0).sum():,}  AUC={fold_auc:.4f}')

valid = ~np.isnan(oof_posterior)
auc_oof = roc_auc_score(labels[valid], oof_posterior[valid])

print(f'\n{"="*50}')
print(f'Out-of-fold AUC: {auc_oof:.4f}')
print(f'Fold range:      [{min(fold_aucs):.4f}, {max(fold_aucs):.4f}]')
print(f'{"="*50}')

## Step 6 — ROC curves and baselines

Compare the 3-channel model against each channel alone to see what the combination buys.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Panel 1: ROC curves ──
ax = axes[0]

# 3-channel model (out-of-fold)
fpr, tpr, _ = roc_curve(labels[valid], oof_posterior[valid])
ax.plot(fpr, tpr, 'darkorange', lw=2.5, label=f'3-channel Bayesian (AUC={auc_oof:.3f})')

# Baselines: raw feature values (no fitting needed — just rank by the feature)
baselines = [
    ('entropy_similarity', 'steelblue',  1),   # higher = more likely TP
    ('sim_gap',            'seagreen',   1),
    ('signed_delta_rt',    'mediumpurple', -1), # smaller |RT error| = more likely TP
]

for col, color, direction in baselines:
    vals = top1[col].values * direction
    if col == 'signed_delta_rt':
        vals = -np.abs(top1[col].values)  # use -|delta_rt| for ranking
        display_name = '-|signed_delta_rt|'
    else:
        display_name = col
    auc_b = roc_auc_score(labels, vals)
    fpr_b, tpr_b, _ = roc_curve(labels, vals)
    ax.plot(fpr_b, tpr_b, color=color, lw=1.5, alpha=0.7,
            label=f'{display_name} (AUC={auc_b:.3f})')

ax.plot([0, 1], [0, 1], 'k--', lw=0.8, alpha=0.4)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC: 3-channel model vs individual features')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.2)

# ── Panel 2: channel ablation ──
ax = axes[1]

ablation = {'Full model': auc_oof}
for skip_name in ['entropy_sim', 'sim_gap', 'signed_delta_rt']:
    channels_abl = [c for c in CHANNELS if c.name != skip_name]
    oof_abl = np.full(len(top1), np.nan)
    for train_idx, test_idx in gkf.split(top1, labels, groups):
        fitted_abl = []
        for spec in channels_abl:
            fc = fit_channel(spec, top1.iloc[train_idx][spec.feature_col].values,
                           labels[train_idx])
            if fc is not None:
                fitted_abl.append(fc)
        lr_abl = np.zeros(len(test_idx))
        for fc in fitted_abl:
            lr_abl += fc.logLR(top1.iloc[test_idx][fc.spec.feature_col].values)
        oof_abl[test_idx] = 1.0 / (1.0 + np.exp(-(prior_log_odds + lr_abl)))
    v = ~np.isnan(oof_abl)
    ablation[f'w/o {skip_name}'] = roc_auc_score(labels[v], oof_abl[v])

names = list(ablation.keys())
vals = list(ablation.values())
colors = ['darkorange', 'steelblue', 'seagreen', 'mediumpurple']
bars = ax.barh(range(len(names)), vals, color=colors)
ax.set_yticks(range(len(names)))
ax.set_yticklabels(names)
ax.set_xlabel('Out-of-fold AUC')
ax.set_title('Channel ablation — what does each channel contribute?')
ax.set_xlim(0.6, 0.9)
for i, v in enumerate(vals):
    delta = v - auc_oof if i > 0 else 0
    label = f'{v:.3f}' if i == 0 else f'{v:.3f} ({delta:+.3f})'
    ax.text(v + 0.003, i, label, va='center', fontsize=10)
ax.axvline(auc_oof, color='darkorange', linestyle='--', lw=1, alpha=0.5)
ax.invert_yaxis()

plt.tight_layout()
plt.show()

## Step 7 — FDR at confidence thresholds

For a given threshold `t`, the **false discovery rate** is:

```
FDR(t) = #{wrong top-1 with posterior >= t} / #{all spectra with posterior >= t}
```

This answers: "if I call everything above threshold t as confident, what fraction is wrong?"

In [ ]:
thresholds = np.linspace(0.01, 0.99, 200)
lab_v = labels[valid]
post_v = oof_posterior[valid]

fdr_curve = []
n_called_curve = []
for t in thresholds:
    called = post_v >= t
    n_called = called.sum()
    if n_called == 0:
        fdr_curve.append(0)
        n_called_curve.append(0)
        continue
    n_fp = ((lab_v == 0) & called).sum()
    fdr_curve.append(n_fp / n_called)
    n_called_curve.append(n_called)

fdr_curve = np.array(fdr_curve)
n_called_curve = np.array(n_called_curve)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Panel 1: FDR vs threshold ──
ax = axes[0]
ax.plot(thresholds, fdr_curve, 'darkorange', lw=2)
ax.axhline(0.05, color='gray', linestyle=':', lw=1, label='5% FDR')
ax.axhline(0.10, color='gray', linestyle='--', lw=1, label='10% FDR')
ax.set_xlabel('Confidence threshold')
ax.set_ylabel('FDR')
ax.set_title('FDR vs confidence threshold (out-of-fold)')
ax.set_xlim(0, 1)
ax.set_ylim(0, 0.5)
ax.legend()
ax.grid(True, alpha=0.2)

# Mark key thresholds
for t in [0.5, 0.7, 0.8, 0.9]:
    idx = np.argmin(np.abs(thresholds - t))
    ax.annotate(f't={t}\nFDR={fdr_curve[idx]:.1%}\nn={n_called_curve[idx]}',
                xy=(t, fdr_curve[idx]), fontsize=8,
                xytext=(t - 0.08, fdr_curve[idx] + 0.06),
                arrowprops=dict(arrowstyle='->', color='gray', lw=0.8))

# ── Panel 2: n called vs threshold ──
ax = axes[1]
ax.plot(thresholds, n_called_curve, 'steelblue', lw=2)
ax.set_xlabel('Confidence threshold')
ax.set_ylabel('# spectra called')
ax.set_title('Coverage vs confidence threshold')
ax.set_xlim(0, 1)
ax.grid(True, alpha=0.2)

# Secondary axis: fraction of total
ax2 = ax.twinx()
ax2.plot(thresholds, n_called_curve / len(lab_v), 'steelblue', lw=0, alpha=0)
ax2.set_ylabel('Fraction of spectra')

plt.tight_layout()
plt.show()

# Summary table
print(f'{"Threshold":>10}  {"Called":>7}  {"Coverage":>9}  {"TP":>6}  {"FP":>6}  {"FDR":>7}')
print('-' * 55)
for t in [0.5, 0.6, 0.7, 0.8, 0.9, 0.95]:
    called = post_v >= t
    n = called.sum()
    n_tp = ((lab_v == 1) & called).sum()
    n_fp = ((lab_v == 0) & called).sum()
    fdr = n_fp / n if n > 0 else 0
    print(f'{t:>10.2f}  {n:>7,}  {n/len(lab_v):>8.1%}  {n_tp:>6,}  {n_fp:>6,}  {fdr:>7.1%}')

## Step 8 — Inspect individual spectra

Pick specific spectra to trace how the model arrives at its score.
This is the best way to build intuition for what drives confidence up or down.

In [ ]:
def explain_spectrum(idx):
    """Trace the model's reasoning for one spectrum."""
    row = top1.iloc[idx]
    label = 'CORRECT' if row['hit_label'] == 1 else 'WRONG'
    post = oof_posterior[idx]
    total_lr = oof_logLR[idx]

    print(f'Spectrum: {row["wiki_id"]}  |  Top-1: {row.get("name", "?")}  |  Label: {label}')
    print(f'Posterior: {post:.3f}  |  Total logLR: {total_lr:+.2f}')
    print(f'Prior log-odds: {prior_log_odds:+.3f}  →  Post log-odds: {prior_log_odds + total_lr:+.3f}')
    print()

    print(f'{"Channel":<20}  {"Raw value":>12}  {"Transformed":>12}  {"logLR":>8}  {"Interpretation"}')
    print('-' * 80)

    for fc in fitted_channels:
        raw_val = row[fc.spec.feature_col]
        lr_val = fc.logLR(np.array([raw_val]))[0]

        if fc.spec.transform is not None:
            t_val = fc.spec.transform(np.array([raw_val]))[0]
            t_str = f'{t_val:.3f}'
        else:
            t_str = '—'

        if lr_val > 1:
            interp = 'strong TP evidence'
        elif lr_val > 0.3:
            interp = 'moderate TP evidence'
        elif lr_val > -0.3:
            interp = 'ambiguous'
        elif lr_val > -1:
            interp = 'moderate FP evidence'
        else:
            interp = 'strong FP evidence'

        print(f'{fc.spec.name:<20}  {raw_val:>12.3f}  {t_str:>12}  {lr_val:>+8.3f}  {interp}')

    print()

# Show examples: highest confidence TP, lowest confidence TP, highest confidence FP
print('=== Highest confidence correct call ===')
tp_mask = (labels == 1) & valid
best_tp_idx = np.where(tp_mask)[0][np.argmax(oof_posterior[tp_mask])]
explain_spectrum(best_tp_idx)

print('=== Lowest confidence correct call (these get flagged) ===')
worst_tp_idx = np.where(tp_mask)[0][np.argmin(oof_posterior[tp_mask])]
explain_spectrum(worst_tp_idx)

print('=== Highest confidence WRONG call (false confidence — the dangerous cases) ===')
fp_mask = (labels == 0) & valid
worst_fp_idx = np.where(fp_mask)[0][np.argmax(oof_posterior[fp_mask])]
explain_spectrum(worst_fp_idx)

## Step 9 — Explore: look up any spectrum by wiki_id

Change the `WIKI_ID` below to trace the model's reasoning for any spectrum in the dataset.

In [ ]:
# ── Change this to look up any spectrum ──
WIKI_ID = top1['wiki_id'].iloc[0]  # replace with e.g. 'a6ICAHB/1018'

match = top1[top1['wiki_id'] == WIKI_ID]
if len(match) == 0:
    print(f'wiki_id "{WIKI_ID}" not found in top-1 table')
else:
    idx = match.index[0]
    explain_spectrum(idx)

    # Also show all candidates for this spectrum (from full feature table)
    all_cands = (ft[ft['wiki_id'] == WIKI_ID]
                 .sort_values('entropy_similarity', ascending=False)
                 .head(10))
    print(f'Top candidates for {WIKI_ID}:')
    print(f'{"Rank":>4}  {"Name":<35}  {"esim":>6}  {"sim_gap":>8}  {"delta_rt":>9}  {"label":>5}')
    print('-' * 75)
    for i, (_, c) in enumerate(all_cands.iterrows()):
        name = str(c.get('name', '?'))[:35]
        rt = f'{c["signed_delta_rt"]:.1f}' if pd.notna(c['signed_delta_rt']) else 'N/A'
        print(f'{i+1:>4}  {name:<35}  {c["entropy_similarity"]:>6.3f}  '
              f'{c["sim_gap"]:>+8.3f}  {rt:>9}  {int(c["hit_label"]):>5}')

## Step 10 — The logit transform (why it matters)

`entropy_similarity` lives in [0, 1] and piles up near 1.0 for both TP and FP.
Fitting a Normal directly on [0, 1] is a bad idea — densities get crushed at the boundaries.

`logit_upper_half` maps [0.5, 1.0] → ℝ, spreading out the high-similarity region where
discrimination happens. But **uncapped**, sim=1.0 maps to logit=+13, which falls far
outside the fitted Normal (mu ~ 1.2, sigma ~ 0.6) and produces a massive, unreliable logLR.

The cap at ±4 prevents this. Below we show what happens with and without the cap.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sim_grid = np.linspace(0.5, 1.0, 500)

# ── Panel 1: raw similarity → logit mapping ──
ax = axes[0]

def logit_uncapped(x):
    x = np.clip(x, 0.5, 1.0 - 1e-6)
    t = (x - 0.5) / 0.5
    t = np.clip(t, 1e-6, 1.0 - 1e-6)
    return np.log(t / (1 - t))

logit_raw = logit_uncapped(sim_grid)
logit_cap = logit_upper_half(sim_grid)

ax.plot(sim_grid, logit_raw, 'gray', lw=1.5, label='uncapped logit')
ax.plot(sim_grid, logit_cap, 'darkorange', lw=2, label='capped at ±4')
ax.axhline(4, color='red', linestyle=':', lw=1, alpha=0.5)
ax.axhline(-4, color='red', linestyle=':', lw=1, alpha=0.5)
ax.set_xlabel('entropy_similarity')
ax.set_ylabel('logit_upper_half')
ax.set_title('Transform: [0.5, 1.0] → logit space')
ax.legend()

# Mark key similarities
for s, label in [(0.7, '0.7'), (0.9, '0.9'), (0.99, '0.99'), (1.0, '1.0')]:
    y_raw = logit_uncapped(np.array([s]))[0]
    y_cap = logit_upper_half(np.array([s]))[0]
    ax.annotate(f'sim={label}\nraw={y_raw:.1f}\ncap={y_cap:.1f}',
                xy=(s, y_cap), fontsize=7, ha='center',
                xytext=(s, y_cap - 2.5),
                arrowprops=dict(arrowstyle='->', color='gray', lw=0.5))

# ── Panel 2: entropy_sim distribution in raw vs logit space ──
ax = axes[1]
tp_sim = top1.loc[labels == 1, 'entropy_similarity'].values
fp_sim = top1.loc[labels == 0, 'entropy_similarity'].values

ax.hist(tp_sim, bins=50, density=True, alpha=0.4, color='steelblue',
        label='TP', range=(0.5, 1.0))
ax.hist(fp_sim, bins=50, density=True, alpha=0.4, color='salmon',
        label='FP', range=(0.5, 1.0))
ax.set_xlabel('entropy_similarity (raw)')
ax.set_ylabel('Density')
ax.set_title('Raw scale — both pile up near 1.0')
ax.legend()

ax = axes[2]
tp_logit = logit_upper_half(tp_sim)
fp_logit = logit_upper_half(fp_sim)

ax.hist(tp_logit, bins=50, density=True, alpha=0.4, color='steelblue', label='TP')
ax.hist(fp_logit, bins=50, density=True, alpha=0.4, color='salmon', label='FP')
ax.set_xlabel('logit_upper_half (capped)')
ax.set_ylabel('Density')
ax.set_title('Logit scale — separation becomes visible')
ax.legend()

plt.tight_layout()
plt.show()

# Show what fraction hits the cap
tp_at_cap = (tp_logit == 4.0).mean()
fp_at_cap = (fp_logit == 4.0).mean()
tp_at_neg = (tp_logit == -4.0).mean()
fp_at_neg = (fp_logit == -4.0).mean()
print(f'Fraction hitting +4 cap: TP={tp_at_cap:.1%}  FP={fp_at_cap:.1%}')
print(f'Fraction hitting -4 cap: TP={tp_at_neg:.1%}  FP={fp_at_neg:.1%}')

## Step 11 — Independence assumption check

The model assumes channels are independent: `p(esim, gap, rt | class) = p(esim|class) × p(gap|class) × p(rt|class)`.

If channels are correlated within a class, the model double-counts evidence. Check with
Spearman correlations within TP and FP separately.

In [ ]:
ch_cols = ['entropy_similarity', 'sim_gap', 'signed_delta_rt']
ch_labels = ['entropy_sim', 'sim_gap', 'signed_delta_rt']

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (mask, title) in zip(axes, [(labels == 1, 'Within TP'), (labels == 0, 'Within FP')]):
    sub = top1.loc[mask, ch_cols]
    corr = sub.corr(method='spearman')

    im = ax.imshow(corr.values, cmap='RdBu_r', vmin=-1, vmax=1)
    ax.set_xticks(range(len(ch_labels)))
    ax.set_yticks(range(len(ch_labels)))
    ax.set_xticklabels(ch_labels, rotation=30, ha='right')
    ax.set_yticklabels(ch_labels)
    ax.set_title(f'{title} (n={mask.sum():,})')

    for i in range(len(ch_labels)):
        for j in range(len(ch_labels)):
            v = corr.values[i, j]
            color = 'white' if abs(v) > 0.4 else 'black'
            ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=12, color=color)

fig.colorbar(im, ax=axes, shrink=0.8, label='Spearman rho')
plt.suptitle('Channel independence check — low |rho| = good', fontsize=13)
plt.tight_layout()
plt.show()

# Print verdict
for lbl_name, mask in [('TP', labels == 1), ('FP', labels == 0)]:
    sub = top1.loc[mask, ch_cols]
    corr = sub.corr(method='spearman')
    max_off_diag = corr.where(~np.eye(len(ch_cols), dtype=bool)).abs().max().max()
    print(f'{lbl_name}: max off-diagonal |rho| = {max_off_diag:.3f}  '
          f'{"(acceptable)" if max_off_diag < 0.3 else "(warning: correlated channels)"}')